In [0]:
display(spark.sql("LIST 's3://healthcare-raw-data-siddharth/raw/patients/'"))

path,name,size,modification_time
s3://healthcare-raw-data-siddharth/raw/patients/patients_records.csv,patients_records.csv,8399221,1786214620000


In [0]:
# One-time reset: drop existing tables so they get recreated with the
# correct schema and Column Mapping settings. Safe to run — Bronze is
# always fully reproducible from the S3 source, and Silver/Gold are
# fully reproducible from Bronze.
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.bronze.patients")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.silver.patients")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.gold.patient_count_per_hospital")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.gold.hospital_ranking")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.gold.condition_contribution")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.gold.admission_type_distribution")
spark.sql("DROP TABLE IF EXISTS healthcare_catalog.gold.insurance_provider_breakdown")
print("Old tables dropped. Ready for a clean rebuild.")

Old tables dropped. Ready for a clean rebuild.


#Bronze Layer

In [0]:
from pyspark.sql import functions as F

s3_path = "s3://healthcare-raw-data-siddharth/raw/patients/patients_records.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(s3_path)
)

# Bronze = raw data only. No renaming, no casting, no cleaning —
# only traceability metadata is added, which does not alter any
# original column or value.
df_bronze = (
    df_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("patients_records.csv"))
)

spark.sql("CREATE CATALOG IF NOT EXISTS healthcare_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_catalog.bronze")

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")   # allows original column names with spaces
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_catalog.bronze.patients")
)

print("Bronze table written: healthcare_catalog.bronze.patients")
print(f"Rows ingested: {df_bronze.count():,}")
display(df_bronze.limit(5))

Bronze table written: healthcare_catalog.bronze.patients
Rows ingested: 55,500


Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,_ingestion_timestamp,_source_file
Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal,2026-08-10T04:26:04.862Z,patients_records.csv
LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,2026-08-10T04:26:04.862Z,patients_records.csv
DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal,2026-08-10T04:26:04.862Z,patients_records.csv
andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal,2026-08-10T04:26:04.862Z,patients_records.csv
adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal,2026-08-10T04:26:04.862Z,patients_records.csv


In [0]:
display(spark.sql("SELECT COUNT(*) AS row_count FROM healthcare_catalog.bronze.patients"))
display(spark.sql("SELECT * FROM healthcare_catalog.bronze.patients LIMIT 5"))

row_count
55500


Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,_ingestion_timestamp,_source_file
Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal,2026-08-10T04:25:53.200Z,patients_records.csv
LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,2026-08-10T04:25:53.200Z,patients_records.csv
DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal,2026-08-10T04:25:53.200Z,patients_records.csv
andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal,2026-08-10T04:25:53.200Z,patients_records.csv
adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal,2026-08-10T04:25:53.200Z,patients_records.csv


#Silver Layer — Cleaning + SCD Type 2

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("healthcare_catalog.bronze.patients")

# Renaming is a standardization step — it belongs in Silver, not Bronze
renamed_cols = [c.replace(" ", "_") for c in df_bronze.columns]
df_renamed = df_bronze.toDF(*renamed_cols)

df_clean = (
    df_renamed
    # 1. Remove exact duplicate business records
    .dropDuplicates(["Name", "Age", "Gender", "Date_of_Admission", "Hospital", "Doctor"])

    # 2. Drop rows missing critical fields
    .filter(
        F.col("Name").isNotNull() &
        F.col("Date_of_Admission").isNotNull() &
        F.col("Hospital").isNotNull()
    )

    # 3. Standardize text casing
    .withColumn("Name", F.initcap(F.trim(F.col("Name"))))
    .withColumn("Hospital", F.initcap(F.trim(F.col("Hospital"))))
    .withColumn("Doctor", F.initcap(F.trim(F.col("Doctor"))))
    .withColumn("Insurance_Provider", F.initcap(F.trim(F.col("Insurance_Provider"))))
    .withColumn("Medical_Condition", F.initcap(F.trim(F.col("Medical_Condition"))))
    .withColumn("Admission_Type", F.initcap(F.trim(F.col("Admission_Type"))))
    .withColumn("Medication", F.initcap(F.trim(F.col("Medication"))))
    .withColumn("Test_Results", F.initcap(F.trim(F.col("Test_Results"))))
    .withColumn("Gender", F.initcap(F.trim(F.col("Gender"))))
    .withColumn("Blood_Type", F.upper(F.trim(F.col("Blood_Type"))))

    # 4. Fix data types
    .withColumn("Age", F.col("Age").cast("int"))
    .withColumn("Billing_Amount", F.round(F.col("Billing_Amount").cast("double"), 2))
    .withColumn("Room_Number", F.col("Room_Number").cast("int"))

    # 5. Standardize dates
    .withColumn("Date_of_Admission", F.to_date("Date_of_Admission", "yyyy-MM-dd"))
    .withColumn("Discharge_Date", F.to_date("Discharge_Date", "yyyy-MM-dd"))

    # 6. Drop rows where type coercion failed or values are implausible
    .filter(
        F.col("Age").isNotNull() &
        F.col("Date_of_Admission").isNotNull() &
        F.col("Billing_Amount").isNotNull() &
        (F.col("Age") >= 0) & (F.col("Age") <= 120) &
        (F.col("Billing_Amount") >= 0)
    )

    # 7. Dedup on the SCD2 business key so MERGE never sees ambiguous matches
    .dropDuplicates(["Name", "Date_of_Admission"])
)

print(f"Bronze rows: {df_bronze.count():,}")
print(f"Clean rows after Silver rules: {df_clean.count():,}")
display(df_clean.limit(5))

Bronze rows: 55,500
Clean rows after Silver rules: 49,890


Name,Age,Gender,Blood_Type,Medical_Condition,Date_of_Admission,Doctor,Hospital,Insurance_Provider,Billing_Amount,Room_Number,Admission_Type,Discharge_Date,Medication,Test_Results,_ingestion_timestamp,_source_file
Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.28,328,Urgent,2024-02-02,Paracetamol,Normal,2026-08-10T04:25:53.200Z,patients_records.csv
Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.33,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,2026-08-10T04:25:53.200Z,patients_records.csv
Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.1,205,Emergency,2022-10-07,Aspirin,Normal,2026-08-10T04:25:53.200Z,patients_records.csv
Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.78,450,Elective,2020-12-18,Ibuprofen,Abnormal,2026-08-10T04:25:53.200Z,patients_records.csv
Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-white,Aetna,14238.32,458,Urgent,2022-10-09,Penicillin,Abnormal,2026-08-10T04:25:53.200Z,patients_records.csv


#SCD Type 2 via MERGE INTO

In [0]:
from delta.tables import DeltaTable

silver_table_name = "healthcare_catalog.silver.patients"

table_exists = spark.catalog.tableExists(silver_table_name)

if not table_exists:
    spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_catalog.silver")

    df_seed = (
        df_clean
        .withColumn("scd_effective_start_date", F.col("Date_of_Admission"))
        .withColumn("scd_effective_end_date", F.lit(None).cast("date"))
        .withColumn("scd_is_current", F.lit("Y"))
        .withColumn("_silver_processed_timestamp", F.current_timestamp())
    )

    df_seed.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    print(f"Silver table seeded with {df_seed.count():,} rows (first load)")

else:
    silver_table = DeltaTable.forName(spark, silver_table_name)

    merge_condition = """
        target.Name = source.Name
        AND target.Date_of_Admission = source.Date_of_Admission
        AND target.scd_is_current = 'Y'
    """

    (
        silver_table.alias("target")
        .merge(df_clean.alias("source"), merge_condition)
        .whenMatchedUpdate(
            condition="""
                target.Discharge_Date IS DISTINCT FROM source.Discharge_Date
                OR target.Test_Results IS DISTINCT FROM source.Test_Results
                OR target.Billing_Amount IS DISTINCT FROM source.Billing_Amount
            """,
            set={
                "scd_effective_end_date": "current_date()",
                "scd_is_current": "'N'"
            }
        )
        .whenNotMatchedInsert(values={
            "Name": "source.Name",
            "Age": "source.Age",
            "Gender": "source.Gender",
            "Blood_Type": "source.Blood_Type",
            "Medical_Condition": "source.Medical_Condition",
            "Date_of_Admission": "source.Date_of_Admission",
            "Doctor": "source.Doctor",
            "Hospital": "source.Hospital",
            "Insurance_Provider": "source.Insurance_Provider",
            "Billing_Amount": "source.Billing_Amount",
            "Room_Number": "source.Room_Number",
            "Admission_Type": "source.Admission_Type",
            "Discharge_Date": "source.Discharge_Date",
            "Medication": "source.Medication",
            "Test_Results": "source.Test_Results",
            "_ingestion_timestamp": "source._ingestion_timestamp",
            "_source_file": "source._source_file",
            "scd_effective_start_date": "source.Date_of_Admission",
            "scd_effective_end_date": "CAST(NULL AS DATE)",
            "scd_is_current": "'Y'"
        })
        .execute()
    )
    print("MERGE complete — Silver table updated with SCD Type 2 logic")

Silver table seeded with 49,890 rows (first load)


In [0]:
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {silver_table_name}"))
display(spark.sql(f"SELECT scd_is_current, COUNT(*) FROM {silver_table_name} GROUP BY scd_is_current"))

row_count
49890


scd_is_current,COUNT(*)
Y,49890


#Gold Layer — Business Aggregations in Spark SQL

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_catalog.gold")

spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.patient_count_per_hospital AS
SELECT Hospital, COUNT(*) AS patient_count
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Hospital
ORDER BY patient_count DESC
""")

spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.hospital_ranking AS
SELECT
    Hospital,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_by_volume
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Hospital
""")

spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.condition_contribution AS
SELECT
    Medical_Condition,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Medical_Condition
ORDER BY patient_count DESC
""")

spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.admission_type_distribution AS
SELECT
    Admission_Type,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Admission_Type
ORDER BY patient_count DESC
""")

spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.insurance_provider_breakdown AS
SELECT
    Insurance_Provider,
    COUNT(*) AS patient_count,
    ROUND(SUM(Billing_Amount), 2) AS total_billing
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Insurance_Provider
ORDER BY patient_count DESC
""")

print("All Gold tables created.")
display(spark.sql("SELECT * FROM healthcare_catalog.gold.condition_contribution"))
display(spark.sql("SELECT * FROM healthcare_catalog.gold.hospital_ranking ORDER BY rank_by_volume LIMIT 10"))

All Gold tables created.


Medical_Condition,patient_count,avg_billing,pct_of_total
Arthritis,8428,25516.64,16.89
Diabetes,8362,25706.78,16.76
Hypertension,8299,25545.19,16.63
Obesity,8274,25840.78,16.58
Cancer,8274,25281.08,16.58
Asthma,8253,25738.77,16.54


Hospital,patient_count,avg_billing,rank_by_volume
Llc Smith,40,23714.03,1
Ltd Smith,35,26147.62,2
Smith Ltd,35,25648.42,2
Johnson Plc,35,29726.62,2
Smith Plc,33,27313.92,5
Smith Group,31,23878.47,6
Group Smith,30,28248.36,7
Smith Inc,29,22583.68,8
Johnson Inc,28,28847.99,9
Smith Llc,28,21953.91,9
